In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
# 首先就是对模型整体架构进行定义
class yolov1(nn.Module):
    def __init__(self , S , B , C , input_size = 448):
        '''Yolov1 的基本模型架构
        Args:
            S : int : 将图片划分为 S x S 的网格
            B : int : 每个网格预测 B 个边界框
            C : int : 分类的类别数
        Returns:
        
        '''
        super(yolov1 , self).__init__()
        self.s = S
        self.b = B
        self.c = C
        self.size = input_size

        # 特征提取网络部分
        self.feature_extractor = nn.Sequential(
            nn.AdaptiveAvgPool2d((self.size,self.size)), # 平均池化来自适应调整输入图片大小
            nn.Conv2d(3,64,kernel_size=7,stride=2,padding=3), # (K-1)/2=3 , 输出大小为 (64, 224, 224)
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),# 2x2 最大池化 , 输出大小为 (64, 112, 112)

            nn.Conv2d(64 , 192 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (192, 112, 112)
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2 , stride=2), # 输出大小为 (192, 56, 56)

            nn.Conv2d(192 , 128 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (128, 56, 56)
            nn.LeakyReLU(0.1),
            nn.Conv2d(128 , 256 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (256, 56, 56)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 256 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (256, 56, 56)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 512 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (512, 56, 56)
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2 , stride=2), # 输出大小为 (512, 28, 28)

            nn.Conv2d(512 , 256 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (256, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 512 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (512, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 256 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (256, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 512 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (512, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 256 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (256, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 512 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (512, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 256 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (256, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(256 , 512 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (512, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 512 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (512, 28, 28)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 28, 28)
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2 , stride=2), # 输出大小为 (1024, 14, 14)

            nn.Conv2d(1024 , 512 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (512, 14, 14)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 14, 14)
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024 , 512 , kernel_size=1 , stride=1 , padding=0), # 输出大小为 (512, 14, 14)
            nn.LeakyReLU(0.1),
            nn.Conv2d(512 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 14, 14)
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 14, 14)
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024 , 1024 , kernel_size=3 , stride=2 , padding=1), # 输出大小为 (1024, 7, 7)
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 7, 7)
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024 , 1024 , kernel_size=3 , stride=1 , padding=1), # 输出大小为 (1024, 7, 7)
            nn.LeakyReLU(0.1),
        )

        # 全连接层部分
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * self.s * self.s , 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096 , self.s * self.s * (self.c + self.b * 5)),
        )

    def forward(self , x):
        x = self.feature_extractor(x)
        x = self.fc(x)
        x = x.view(-1 , self.s , self.s , self.c + self.b * 5)
        return x

In [3]:
model = yolov1(S=7 , B=2 , C=20)
# print(model)
x = torch.randn((2,3,448,448))
out = model(x)
out.size()

torch.Size([2, 7, 7, 30])

In [4]:
import torch
import torch.nn as nn

class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=20, lambda_coord=5, lambda_noobj=0.5):
        super(YoloLoss, self).__init__()
        self.s = S
        self.b = B
        self.c = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.mse = nn.MSELoss(reduction="sum")

    def compute_iou(self, box1, box2):
        # 统一单位到 Grid Cell
        b1_x = box1[..., 0:1]
        b1_y = box1[..., 1:2]
        b1_w = box1[..., 2:3] * self.s 
        b1_h = box1[..., 3:4] * self.s 

        b2_x = box2[..., 0:1]
        b2_y = box2[..., 1:2]
        b2_w = box2[..., 2:3] * self.s 
        b2_h = box2[..., 3:4] * self.s 

        # 计算角点
        b1_x1, b1_y1 = b1_x - b1_w / 2, b1_y - b1_h / 2
        b1_x2, b1_y2 = b1_x + b1_w / 2, b1_y + b1_h / 2
        
        b2_x1, b2_y1 = b2_x - b2_w / 2, b2_y - b2_h / 2
        b2_x2, b2_y2 = b2_x + b2_w / 2, b2_y + b2_h / 2

        # IOU 计算
        x1 = torch.max(b1_x1, b2_x1)
        y1 = torch.max(b1_y1, b2_y1)
        x2 = torch.min(b1_x2, b2_x2)
        y2 = torch.min(b1_y2, b2_y2)

        intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
        
        box1_area = b1_w * b1_h
        box2_area = b2_w * b2_h
        union = box1_area + box2_area - intersection + 1e-6

        return intersection / union

    def forward(self, predictions, target):
        # predictions : (N, 7, 7, 30)
        
        # 1. 确保掩码是 3 维的 (N, 7, 7)
        Iobj_i = target[..., 4] > 0 
        Inoobj_i = target[..., 4] == 0

        # 2. 找出负责预测的框
        with torch.no_grad():
            target_box = target[..., :4] 
            box1_pred = predictions[..., :4]
            box2_pred = predictions[..., 5:9]
            
            iou1 = self.compute_iou(box1_pred, target_box)
            iou2 = self.compute_iou(box2_pred, target_box)
            
            ious = torch.cat([iou1, iou2], dim=-1)
            iou_max, best_box_idx = torch.max(ious, dim=-1) # (N, 7, 7)

        # 3. 准备预测数据 [重点修正区域]
        # 必须先 reshape 成 (N, 7, 7, 2, 5)，绝对不能把 B=2 放在前面
        pred_boxes = predictions[..., :10].reshape(-1, self.s, self.s, self.b, 5) 
        
        # 扩展索引维度以匹配 gather
        # best_box_idx: (N, 7, 7) -> (N, 7, 7, 1, 5)
        best_box_idx_expanded = best_box_idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, -1, -1, 5)
        
        # 提取负责的框 -> (N, 7, 7, 1, 5) -> squeeze -> (N, 7, 7, 5)
        box_predicted = pred_boxes.gather(3, best_box_idx_expanded).squeeze(3)
        
        # === 调试打印 (如果再次报错，请检查这里打印出的形状) ===
        # print(f"DEBUG: Mask shape: {Iobj_i.shape}")
        # print(f"DEBUG: Box Pred shape: {box_predicted.shape}")
        # 如果 box_predicted 是 [2, 2, 7, 5]，说明上面的 reshape 写错了
        
        box_targets = target[..., :5]

        # 4. 计算损失
        # 4.1 坐标损失
        # 这里的索引操作要求 box_predicted 必须是 (N, 7, 7, 5)
        box_pred_coord = box_predicted[Iobj_i, :2] 
        box_target_coord = box_targets[Iobj_i, :2]
        loss_xy = self.mse(box_pred_coord, box_target_coord)

        # 4.2 宽高损失
        box_pred_wh = box_predicted[Iobj_i, 2:4]
        box_target_wh = box_targets[Iobj_i, 2:4]
        box_pred_wh_sqrt = torch.sqrt(torch.abs(box_pred_wh) + 1e-6)
        box_target_wh_sqrt = torch.sqrt(box_target_wh)
        loss_wh = self.mse(box_pred_wh_sqrt, box_target_wh_sqrt)

        # 4.3 置信度损失 (有物体)
        pred_conf_obj = box_predicted[Iobj_i, 4]
        target_conf_obj = iou_max[Iobj_i].detach() 
        loss_obj = self.mse(pred_conf_obj, target_conf_obj)

        # 4.4 置信度损失 (无物体)
        # 背景网格
        pred_conf_noobj_grid = predictions[Inoobj_i, 4] 
        pred_conf_noobj_grid = torch.cat([pred_conf_noobj_grid, predictions[Inoobj_i, 9]]) 
        loss_noobj = self.mse(pred_conf_noobj_grid, torch.zeros_like(pred_conf_noobj_grid))

        # 有物体网格中的非负责框
        not_best_box_idx = 1 - best_box_idx 
        pred_conf_all = predictions[..., [4, 9]] 
        conf_not_responsible = pred_conf_all.gather(3, not_best_box_idx.unsqueeze(-1)).squeeze(-1)
        conf_not_responsible = conf_not_responsible[Iobj_i]
        loss_noobj += self.mse(conf_not_responsible, torch.zeros_like(conf_not_responsible))

        # 4.5 分类损失
        pred_cls = predictions[Iobj_i, 10:]
        target_cls = target[Iobj_i, 10:]
        loss_cls = self.mse(pred_cls, target_cls)

        total_loss = (
            self.lambda_coord * (loss_xy + loss_wh) +
            loss_obj +
            self.lambda_noobj * loss_noobj +
            loss_cls
        )

        return total_loss



In [5]:
# === 测试脚本 ===
if __name__ == "__main__":
    x = torch.randn((2, 7, 7, 30)) 
    y = torch.zeros((2, 7, 7, 30))
    y[..., :4] = torch.rand((2, 7, 7, 4)) 
    y[..., 4] = 1 
    y[..., 9] = 1 

    criterion = YoloLoss(S=7, B=2, C=20)
    loss = criterion(x, y)
    print("Loss:", loss.item())

IndexError: The shape of the mask [2, 7, 7] at index 1 does not match the shape of the indexed tensor [2, 2, 7, 5] at index 1

In [ ]:
import torch
import torch.optim as optim
from tqdm import tqdm  # 进度条库，pip install tqdm

def train_fn(train_loader, model, optimizer, loss_fn, device):
    loop = tqdm(train_loader, leave=True)
    mean_loss = []

    model.train() # 开启训练模式 (启用 Dropout 和 BatchNorm)

    for batch_idx, (x, y) in enumerate(loop):
        # 1. 搬运数据到 GPU
        x, y = x.to(device), y.to(device)

        # 2. 前向传播
        out = model(x)
        
        # 3. 计算损失
        loss = loss_fn(out, y)
        mean_loss.append(loss.item())

        # 4. 反向传播 (三板斧)
        optimizer.zero_grad() # 清空过往梯度
        loss.backward()       # 计算当前梯度
        optimizer.step()      # 更新权重

        # 5. 更新进度条 (显示当前 Loss)
        loop.set_postfix(loss=loss.item())

    print(f"Mean loss was {sum(mean_loss)/len(mean_loss)}")

TypeError: train_fn() missing 5 required positional arguments: 'train_loader', 'model', 'optimizer', 'loss_fn', and 'device'

In [ ]:
# # 假设你已经写好了 Dataset 并命名为 VOCDataset
# # from dataset import VOCDataset 
# from torch.utils.data import DataLoader

# # 超参数设置
# LEARNING_RATE = 2e-5 # YOLOv1 刚开始训练时 LR 要小，否则容易飞
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 16 # 根据你显存大小调整 (16, 32, 64)
# WEIGHT_DECAY = 0
# EPOCHS = 100
# NUM_WORKERS = 2
# PIN_MEMORY = True

# def main():
#     # 1. 初始化模型
#     model = YOLOv1(S=7, B=2, C=20).to(DEVICE)
    
#     # 2. 初始化优化器
#     optimizer = optim.Adam(
#         model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
#     )
    
#     # 3. 初始化损失函数
#     loss_fn = YoloLoss()

#     # 4. 准备数据 (这里假设你已经做好了 Dataset)
#     # train_dataset = VOCDataset("train.csv", transform=transform)
#     # train_loader = DataLoader(
#     #     dataset=train_dataset,
#     #     batch_size=BATCH_SIZE,
#     #     num_workers=NUM_WORKERS,
#     #     pin_memory=PIN_MEMORY,
#     #     shuffle=True,
#     #     drop_last=True
#     # )

#     # 模拟一个 loader 方便你测试代码跑通 (实际跑的时候请删掉下面这两行，解开上面的注释)
#     # 随机生成数据: 16张图, 3通道, 448x448
#     fake_data = torch.randn(2, 3, 448, 448) 
#     # 随机生成标签: 2个样本, 7x7网格, 30个值
#     fake_target = torch.zeros(2, 7, 7, 30) 
#     # 假装有些格子里有物体 (置信度设为1)
#     fake_target[:, 3, 3, 4] = 1 
#     # 封装成简单的 list 模拟 loader
#     train_loader = [(fake_data, fake_target)] * 10 

#     # 5. 开始循环训练
#     for epoch in range(EPOCHS):
#         print(f"Epoch: {epoch+1}/{EPOCHS}")
#         train_fn(train_loader, model, optimizer, loss_fn, DEVICE)
        
#         # 每一轮结束，你可以保存一下模型
#         if (epoch + 1) % 10 == 0:
#             torch.save(model.state_dict(), f"yolov1_epoch_{epoch+1}.pth")
#             print("Model saved!")

# if __name__ == "__main__":
#     main()